# 🚀 VLA 연구 · 9회차 실습
## VLM 기본 원리 & MuJoCo 실시간 분류 파이프라인

> **오늘 목표**  
> 1. VLM(Vision-Language Model)의 아키텍처 원리를 이해한다.  
> 2. SmolVLM을 직접 로드하고 이미지를 분석해 본다.  
> 3. MuJoCo 씬 → VLM 행동 분류 파이프라인을 완성한다.  

> **오늘 사용하는 핵심 모델**: `HuggingFaceTB/SmolVLM-Instruct`  
> → 이 모델은 나중에 파인튜닝할 **smoVLA의 비전 백본**입니다. 최종 프로젝트에 직접 활용할 수 있습니다.


>시각화 시 사용할 한글 폰트 설정 (아래 블록 실행 후 **런타임 재시작**)

In [ ]:
# 한글 폰트 설정 (Colab 환경)
import matplotlib
import matplotlib.font_manager as fm
import subprocess
subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], capture_output=True)
fm._load_fontmanager(try_read_cache=False)
matplotlib.rcParams['font.family'] = 'NanumGothic'
matplotlib.rcParams['axes.unicode_minus'] = False
print('폰트 설정 완료 — 런타임을 재시작하세요.')


---
## SECTION 0  |  패키지 설치 & 환경 확인

> ⚠️ **처음 실행 시 두 번의 런타임 재시작이 필요합니다**  
> 1. 폰트 설치 셀 실행 → **런타임 재시작** → 타이틀 셀부터 재실행  
> 2. 패키지 설치 셀(0-1) 실행 → **런타임 재시작** → 0-2 셀부터 재실행  
> 두 번째 이후 실행 시엔 재시작 없이 순서대로 실행해도 됩니다.


In [ ]:
# 0-1. 패키지 설치 — 이 셀 실행 후 Runtime > Restart session
# (처음 1회만 필요 / Pillow 충돌 방지를 위해 --force-reinstall 사용)
import os
os.environ['MUJOCO_GL'] = 'egl'

# SmolVLM 최소 요구 버전: transformers>=4.49.0
!pip install -q 'transformers>=4.49.0' accelerate
!pip install -q --force-reinstall 'Pillow>=10.4.0,<12.0'
!pip install -q mujoco mediapy imageio

# 설치 결과 확인
import transformers, PIL
print(f'✓ transformers : {transformers.__version__}')
print(f'✓ Pillow       : {PIL.__version__}')
print()
print('⚠️  Runtime > Restart session 으로 재시작 후 다음 셀(0-2)부터 실행하세요.')


In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception as e:
    IN_COLAB = False
    print('Colab 환경이 아니거나 Drive 마운트가 불가능합니다.')
    print(f'상세: {e}')

DRIVE_BASE = '/content/drive/MyDrive/VLA_2026'
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f'Drive 작업 경로: {DRIVE_BASE}')

In [ ]:
# 0-2. 하네스 로드 & 환경 확인 (런타임 재시작 후 이 셀부터 실행)
import shutil
shutil.copy('/content/drive/MyDrive/VLA_Team_2026/nmail_harness.py', '/content/')
import nmail_harness as H
device = H.setup()


In [ ]:
# 0-3. 라이브러리 임포트
import os
os.environ.setdefault('MUJOCO_GL', 'egl')

import mujoco
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch

print(f'✓ MuJoCo  : {mujoco.__version__}')
print(f'✓ torch   : {torch.__version__}')
import PIL; print(f'✓ Pillow  : {PIL.__version__}')
import transformers; print(f'✓ transformers: {transformers.__version__}')


---
## SECTION 1  |  VLM 아키텍처 원리

### 1-1. VLM 이란?

**VLM(Vision-Language Model)** 은 이미지와 텍스트를 *동시에* 이해하는 모델입니다.  
이미지를 "읽고" 자연어로 답변하거나, 이미지를 보고 행동을 결정할 수 있습니다.

```
[이미지] → [Vision Encoder (ViT)] → [이미지 토큰 임베딩]
                                            ↓
[텍스트 토큰] ─────────────────→ [Language Model (LLM)] → [응답 텍스트]
```

| 구성 요소 | 역할 | 예시 |
|-----------|------|------|
| Vision Encoder | 이미지를 고정 크기 패치로 분할 → 임베딩 벡터 시퀀스 생성 | SigLIP-400M |
| Pixel Shuffle | 비전 임베딩 토큰 수를 1/4로 압축 (효율성) | 4×4 → 1 |
| Language Model | 이미지 토큰 + 텍스트 토큰을 함께 처리 | SmolLM2-2B |

### 1-2. SmolVLM 아키텍처

```
SmolVLM-Instruct (2B)
├── Vision Encoder : SigLIP-400M  (이미지 → 1024-dim 패치 토큰)
├── Pixel Shuffle  : 4×4 패치를 1개 토큰으로 압축
└── LLM Backbone   : SmolLM2-2B   (텍스트 생성)
```

**왜 SmolVLM인가?**  
- 2B 파라미터 → T4 GPU (16GB)에서 실행 가능  
- smoVLA의 비전 백본과 동일 → 12회차 파인튜닝과 직접 연결  
- 오픈소스 → 직접 가중치 수정 가능

---

### 1-3. CLIP이란? — VLM과의 관계

**CLIP(Contrastive Language-Image Pre-training, OpenAI 2021)** 은 VLM의 *전신*에 해당하는 모델입니다.  
VLM을 이해하려면 CLIP의 아이디어를 먼저 파악해두면 좋습니다.

**핵심 아이디어**: 이미지와 텍스트를 *같은 벡터 공간*에 임베딩한다.

```
[이미지] → [Image Encoder] → 이미지 벡터 ─┐
                                            ├─ cosine 유사도 → 점수
[텍스트] → [Text Encoder]  → 텍스트 벡터 ─┘
```

**학습 방식 (Contrastive Learning)**:
- 4억 개의 (이미지, 텍스트) 쌍으로 학습
- "같은 쌍"의 유사도는 높이고, "다른 쌍"의 유사도는 낮추는 방식
- 결과: 이미지 "고양이 사진" ↔ 텍스트 "a photo of a cat" → 높은 유사도

**CLIP의 한계와 VLM의 등장**:

| | CLIP | SmolVLM (오늘) |
|--|---|---|
| 출력 형식 | 유사도 **점수** (숫자) | 자유형식 **텍스트** |
| 할 수 있는 것 | 미리 정의된 레이블 중 선택 | 복잡한 장면 묘사·추론 |
| 이미지-텍스트 결합 방식 | 별도 인코더, 유사도 비교 | 통합 Encoder-Decoder |
| 행동 분류 방법 | 코사인 유사도로 가장 가까운 레이블 선택 | 프롬프트 → 텍스트 파싱 |
| 장점 | 빠름, 별도 생성 불필요 | 복잡한 추론 가능, 설명 포함 |
| 단점 | 레이블 집합이 고정됨 | 속도 느림, 모델 크기 큼 |

> **정리**: CLIP은 "이미지와 텍스트가 얼마나 닮았나?"를 묻고,  
> VLM은 "이 이미지를 보고 무슨 말을 할 수 있나?"를 묻습니다.  
> 우리 프로젝트는 장면 설명과 추론이 필요하므로 VLM이 더 적합합니다.


In [ ]:
# SmolVLM 모델 및 프로세서 로드 (최초 실행시 약 30~60초 소요)
# SmolVLM 공식 요구사항: transformers >= 4.49.0
from transformers import AutoProcessor

try:
    from transformers import AutoModelForVision2Seq
    _ModelCls = AutoModelForVision2Seq
except ImportError:
    # transformers < 4.49 환경 — 구체 클래스로 대체
    from transformers import Idefics3ForConditionalGeneration
    _ModelCls = Idefics3ForConditionalGeneration
    print('⚠️  AutoModelForVision2Seq 없음 — Idefics3ForConditionalGeneration으로 대체')

MODEL_ID = 'HuggingFaceTB/SmolVLM-Instruct'
processor = AutoProcessor.from_pretrained(MODEL_ID)

print(f'Vision Processor 설정:')
print(f'  processor 타입  : {type(processor).__name__}')
print(f'  image_processor : {type(processor.image_processor).__name__}')
print(f'  최대 이미지 크기 : {processor.image_processor.size}')


In [ ]:
# SmolVLM 모델 로드 (약 30~60초, 첫 실행 시 모델 다운로드 포함)
vlm_model = _ModelCls.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
vlm_model.eval()

# pad_token_id 수정
# 모델 config의 pad_token_id(128002)가 vocab 크기(32000)를 초과 → eos_token_id로 대체
eos_id = processor.tokenizer.eos_token_id
vlm_model.config.pad_token_id = eos_id
vlm_model.generation_config.pad_token_id = eos_id

total_params = sum(p.numel() for p in vlm_model.parameters()) / 1e9
print(f'✓ SmolVLM 로드 완료')
print(f'  클래스          : {type(vlm_model).__name__}')
print(f'  총 파라미터     : {total_params:.2f}B')
print(f'  실행 디바이스   : {next(vlm_model.parameters()).device}')
print(f'  pad_token_id    : {vlm_model.config.pad_token_id} (eos_token_id)')


---
## SECTION 2  |  SmolVLM 기본 사용

### 2-1. 이미지 분석 함수 정의

VLM에 이미지를 입력하는 과정:  
1. 이미지를 PIL 또는 numpy 배열로 준비  
2. `processor`로 이미지+텍스트를 토큰화  
3. `model.generate()`로 응답 생성  
4. `processor.decode()`로 텍스트 복원  


In [ ]:
def smolvlm_query(img_np, prompt_text):
    """numpy 이미지 배열과 프롬프트 텍스트를 받아 VLM 응답 문자열을 반환"""
    pil_img = Image.fromarray(img_np)
    messages = [{'role': 'user', 'content': [
        {'type': 'image'},
        {'type': 'text', 'text': prompt_text}
    ]}]
    prompt_str = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=prompt_str, images=[pil_img], return_tensors='pt')
    inputs = {k: v.to(vlm_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        generated_ids = vlm_model.generate(**inputs, max_new_tokens=256, do_sample=False)

    new_tokens = generated_ids[0, inputs['input_ids'].shape[1]:]
    return processor.decode(new_tokens, skip_special_tokens=True).strip()

print('✓ smolvlm_query 함수 등록 완료')


In [ ]:
# 테스트 이미지 생성 — 간단한 도형으로 확인
import matplotlib.patches as patches
import io

fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.add_patch(patches.FancyBboxPatch((0.25, 0.25), 0.5, 0.5,
    boxstyle='round,pad=0.02', facecolor='tomato', edgecolor='darkred', lw=2))
ax.text(0.5, 0.5, 'BOX', fontsize=20, ha='center', va='center',
        fontweight='bold', color='white')
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
ax.set_title('테스트 이미지: 빨간 박스')

buf = io.BytesIO()
fig.savefig(buf, format='png', bbox_inches='tight', facecolor='white')
buf.seek(0)
test_img = np.array(Image.open(buf).convert('RGB'))
plt.show()
print(f'이미지 shape: {test_img.shape}')


In [ ]:
# 기본 질문으로 VLM 응답 확인
response = smolvlm_query(test_img, 'What do you see in this image? Describe in one sentence.')
print('VLM 응답:', response)


In [ ]:
# 행동 분류 프롬프트 적용
response_action = smolvlm_query(test_img, H.PROMPT_ACTION_JSON)
print('행동 분류 프롬프트:')
print(H.PROMPT_ACTION_JSON)
print("--------------------------------")
print('응답 (raw):', response_action)
print()
action = H.parse_action(response_action)
print(f'파싱된 행동: [{action}]  (예상: pick)')


---
## SECTION 3  |  MuJoCo 씬 분석

### 3-1. 기숙사 로비 씬 4종 렌더링

최종 프로젝트의 4가지 행동 타입에 맞는 씬을 정의하고 렌더링합니다.


In [ ]:
# 3-0. Menagerie Panda 다운로드 & 렌더링 헬퍼
import subprocess
from pathlib import Path

WORK_DIR = Path('/content')
MENAGERIE_DIR = WORK_DIR / 'mujoco_menagerie'
if not MENAGERIE_DIR.exists():
    print('MuJoCo Menagerie clone 중...')
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/google-deepmind/mujoco_menagerie.git',
         str(MENAGERIE_DIR)],
        check=True, capture_output=True)
    print('✓ clone 완료')
else:
    print('✓ 이미 Menagerie 폴더가 있습니다.')
PANDA_DIR = MENAGERIE_DIR / 'franka_emika_panda'
print('PANDA_DIR:', PANDA_DIR)

def render_panda_scene(xml_content, scene_name='scene', camera='overview',
                       width=320, height=240, steps=200,
                       robot_pos=(0.0, -0.7, 0.40), robot_yaw_deg=90.0):
    """Panda <include> XML을 PANDA_DIR에 저장→로드→렌더링.

    robot_pos      : (x, y, z) — 로봇 베이스 위치 (테이블 표면 z=0.40)
    robot_yaw_deg  : Z축 회전(도).  90 → 로봇이 +Y(테이블 뒤쪽)를 바라봄
    """
    scene_path = PANDA_DIR / f'_tmp_{scene_name}.xml'
    scene_path.write_text(xml_content, encoding='utf-8')
    model = mujoco.MjModel.from_xml_path(str(scene_path))
    data  = mujoco.MjData(model)

    # 로봇 베이스 위치·방향 설정 (include 후 프로그래밍으로 조정)
    for bname in ('link0', 'panda_link0'):
        bid = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, bname)
        if bid >= 0:
            model.body_pos[bid]  = list(robot_pos)
            yaw = np.deg2rad(robot_yaw_deg)
            # Z축 회전 쿼터니언: [cos(θ/2), 0, 0, sin(θ/2)]
            model.body_quat[bid] = [np.cos(yaw / 2), 0.0, 0.0, np.sin(yaw / 2)]
            break

    # Panda 홈 포즈 (keyframe 우선)
    if model.nkey > 0:
        mujoco.mj_resetDataKeyframe(model, data, 0)
    else:
        mujoco.mj_resetData(model, data)

    for _ in range(steps):
        mujoco.mj_step(model, data)

    renderer = mujoco.Renderer(model, height=height, width=width)
    renderer.update_scene(data, camera=camera)
    img = renderer.render().copy()
    renderer.close()
    return img

print('✓ render_panda_scene 등록 완료')


In [ ]:
# 씬 타입별 XML 정의 (Panda 로봇 포함)
# 좌표 규칙: Y- = 테이블 앞(로봇·카메라 방향), Y+ = 테이블 뒤(물체 배치)
BASE_XML = '''
<mujoco model="lobby_scene">
  <include file="panda.xml"/>
  <option timestep="0.005" gravity="0 0 -9.81"/>
  <visual>
    <headlight diffuse="0.4 0.4 0.4" ambient="0.2 0.2 0.2" specular="0.3 0.3 0.3"/>
  </visual>
  <worldbody>
    <light name="key" pos="0 -0.5 2.8" dir="0 0.2 -1" directional="true"/>
    <geom name="floor" type="plane" size="10 10 0.05" rgba="0.82 0.84 0.86 1"/>
    <!-- 테이블: Y 앞쪽(-0.80)을 확장해 로봇(Y≈-0.45) 탑재 공간 확보 -->
    <geom name="table" type="box" pos="0 -0.15 0.35" size="0.55 0.8 0.05"
          rgba="0.65 0.50 0.35 1" contype="1" conaffinity="1"/>
    {target}
    {obstacles}
    <!-- 기울어진 top-view: 오른쪽+위에서 내려다보기 -->
    <camera name="overview" pos="0.8 -1.5 1.8" xyaxes="0.85 0.53 0 -0.35 0.56 0.75"/>
  </worldbody>
</mujoco>
'''

# 물체는 테이블 뒤쪽(Y > 0)에 배치 — 로봇이 +Y를 바라보며 조작
TARGET = (
    '<geom name="target" type="box" pos="0.2 0.3 0.50" '
    'size="0.10 0.10 0.07" rgba="0.9 0.15 0.1 1" contype="1" conaffinity="1"/>'
)

SCENE_CONFIGS = {
    'pick': {
        'target': TARGET,
        'obstacles': '',
    },
    'push': {
        'target': TARGET,
        'obstacles': (
            '<geom name="obs1" type="cylinder" pos="0.15 0.02 0.49" '
            'size="0.1 0.25" rgba="0.2 0.6 0.9 1" contype="1" conaffinity="1"/>'
        ),
    },
    'reorder': {
        'target': TARGET,
        'obstacles': (
            '<geom name="obs1" type="cylinder" pos="0.15 0.02 0.49" '
            'size="0.1 0.25" rgba="0.2 0.6 0.9 1" contype="1" conaffinity="1"/>'
            '<geom name="obs2" type="box" pos="-0.22 0.20 0.49" '
            'size="0.07 0.07 0.07" rgba="0.2 0.8 0.3 1" contype="1" conaffinity="1"/>'
            '<geom name="obs3" type="sphere" pos="-0.20 -0.1 0.49" '
            'size="0.06" rgba="1.0 0.8 0.1 1" contype="1" conaffinity="1"/>'
        ),
    },
    'ignore': {
        'target': '',
        'obstacles': '',
    },
}

# 렌더링 (render_panda_scene 사용)
scene_imgs = {}
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (label, cfg) in zip(axes, SCENE_CONFIGS.items()):
    xml = BASE_XML.format(**cfg)
    img = render_panda_scene(xml, scene_name=f'scene_{label}', robot_pos=(0.0, -0.7, 0.40))
    scene_imgs[label] = img
    ax.imshow(img)
    ax.set_title(f'행동: {label}', fontsize=12, fontweight='bold')
    ax.axis('off')
plt.suptitle('4가지 기본 씬 타입 (Panda 로봇 포함)', fontsize=14)
plt.tight_layout(); plt.show()


### 3-2. VLM으로 MuJoCo 씬 분류

이제 실제 MuJoCo 렌더링 이미지를 SmolVLM에 입력합니다.  
이것이 **계층적 파이프라인의 1단계** (VLM 행동 분류) 입니다.


In [ ]:
# 4가지 씬 각각에 대해 VLM 분류 실행
results_basic = []
for gt_label, img in scene_imgs.items():
    response = smolvlm_query(img, H.PROMPT_ACTION_JSON)
    pred = H.parse_action(response)
    correct = (pred == gt_label)
    results_basic.append({'gt': gt_label, 'pred': pred, 'correct': correct, 'response': response})
    print(f"{'✓' if correct else '✗'} GT: {gt_label:8s} | 예측: {pred:8s} | {response[:70]}")

acc = sum(r['correct'] for r in results_basic) / len(results_basic)
print(f'\n정확도: {acc:.0%} ({sum(r["correct"] for r in results_basic)}/{len(results_basic)})')


In [ ]:
# 결과 시각화
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, r, img in zip(axes, results_basic, scene_imgs.values()):
    ax.imshow(img)
    color = 'green' if r['correct'] else 'red'
    ax.set_title(f"GT: {r['gt']}\n예측: {r['pred']}", color=color, fontsize=11, fontweight='bold')
    for spine in ax.spines.values(): spine.set_edgecolor(color); spine.set_linewidth(3)
    ax.axis('off')
plt.suptitle(f'VLM 행동 분류 결과 (정확도 {acc:.0%})', fontsize=14)
plt.tight_layout(); plt.show()


정확도를 높이려면 어떻게 해야 할까요?


---
## SECTION 4  |  프롬프트 엔지니어링

### 4-1. 왜 프롬프트가 중요한가?

같은 모델, 같은 이미지라도 **프롬프트 전략**에 따라 결과가 크게 달라집니다.

| 전략 | 특징 | 장점 | 단점 |
|------|------|------|------|
| Simple | 단순 키워드 요청 | 빠름, 파싱 쉬움 | 맥락 정보 없음 |
| JSON | 구조화 출력 지정 | 파싱 안정적 | 지시 이해 필요 |
| Detailed | 단계별 추론 유도 | 정확도 높음 | 토큰 많이 사용 |

`nmail_harness.py`에 세 가지 프롬프트가 미리 정의되어 있습니다:  
- `H.PROMPT_ACTION_SIMPLE` — 단어 하나만 요구  
- `H.PROMPT_ACTION_JSON` — JSON 형식 응답  
- `H.PROMPT_ACTION_DETAILED` — 단계별 추론 후 JSON


In [ ]:
# 3가지 프롬프트 전략 비교 — push 씬에서 테스트
test_img_push = scene_imgs['push']

prompts = {
    'Simple': H.PROMPT_ACTION_SIMPLE,
    'JSON': H.PROMPT_ACTION_JSON,
    'Detailed': H.PROMPT_ACTION_DETAILED,
}

print ('=== 프롬프트 전략 비교: push 씬 ===\n')
print('사용된 프롬프트:')
for name, prompt in prompts.items():
    print(f'[{name}] {prompt}')
print("\n--------------------------------\n")


print('씬: push (정답: push)\n')
for name, prompt in prompts.items():
    response = smolvlm_query(test_img_push, prompt)
    action = H.parse_action(response)
    print(f'[{name}] 예측: {action:8s} | 응답 앞 100자: {response[:100]}')
    print()


### 4-2. 연습문제 ✏️

**과제**: 아래 셀에 직접 프롬프트를 작성하여 `push` 씬에서 정확히 `push`을 예측하게 만드세요.

**힌트**:
- 모델에게 씬에 무엇이 있는지, 어떤 기준으로 판단해야 하는지 명확히 알려주세요.
- JSON 응답 형식을 요청하면 `H.parse_action()`으로 쉽게 파싱됩니다.
- `H.PROMPT_ACTION_JSON`을 참고하여 구조를 바꿔보세요.


In [ ]:
# ✏️ 연습문제 4-2 — 직접 프롬프트 작성
# push 씬에서 정확히 "push"를 예측하도록 프롬프트를 설계하세요.
#
# 힌트:
#  - 모델에게 명확한 판단 규칙(step-by-step)을 제시하세요
#  - pick / push / reorder / ignore 각각의 조건을 구체적으로 명시하세요
#  - 출력 형식을 고정하세요 (JSON 또는 한 단어)

MY_PROMPT = '''
# TODO: 여기에 프롬프트를 작성하세요
'''

response = smolvlm_query(scene_imgs['push'], MY_PROMPT)
action = H.parse_action(response)
print(f'예측: {action}  (정답: push)')
print(f'응답: {response}')


tip : 프롬프트만으로 해결이 안 된다면, 카메라의 위치, 로봇과 물체의 배치, 물건의 크기와 모양 등을 바꾸어 보세요.

---
## SECTION 5  |  랜덤 씬 파이프라인

### 5-1. 랜덤 씬 생성 함수

도메인 랜덤화를 적용한 씬을 자동으로 생성합니다.  
이 함수는 9회차 '도메인 랜덤화' 세션의 기반이 됩니다.


In [ ]:
import random

def generate_random_scene(action_label=None, seed=None):
    """랜덤 씬 XML과 GT 레이블을 반환합니다.

    Returns:
        (xml_str, gt_label) 튜플
    """
    if seed is not None:
        random.seed(seed)
    if action_label is None:
        action_label = random.choice(['pick', 'push', 'reorder', 'ignore'])

    # 도메인 랜덤화
    floor_r = random.uniform(0.75, 0.92)
    floor_g = random.uniform(0.75, 0.92)
    floor_b = random.uniform(0.70, 0.88)
    # 물체는 테이블 뒤쪽(Y+)에 배치
    target_x = random.uniform(-0.20, 0.20)
    target_y = random.uniform(0.05, 0.25)
    obj_z = 0.50  # 테이블 표면(Z=0.40) + 물체 높이(~0.10)

    target_geom = '' if action_label == 'ignore' else (
        f'<geom name="target" type="box" '
        f'pos="{target_x:.2f} {target_y:.2f} {obj_z:.2f}" '
        f'size="0.10 0.10 0.07" rgba="0.9 0.15 0.1 1" contype="1" conaffinity="1"/>'
    )

    obstacles = ''
    if action_label == 'push':
        r, g, b = random.uniform(0.1, 0.3), random.uniform(0.5, 0.8), random.uniform(0.7, 1.0)
        obs_y = target_y - random.uniform(0.10, 0.20)  # target과 로봇 사이
        obstacles = (
            f'<geom name="obs1" type="cylinder" '
            f'pos="{target_x:.2f} {obs_y:.2f} {obj_z:.2f}" '
            f'size="0.06 0.08" rgba="{r:.2f} {g:.2f} {b:.2f} 1" contype="1" conaffinity="1"/>'
        )
    elif action_label == 'reorder':
        offsets = [(0.0, -0.15), (0.22, 0.10), (-0.22, 0.05)]
        for i, (dx, dy) in enumerate(offsets):
            r, g, b = random.random(), random.random(), random.random()
            obstacles += (
                f'<geom name="obs{i+1}" type="box" '
                f'pos="{target_x+dx:.2f} {target_y+dy:.2f} {obj_z:.2f}" '
                f'size="0.07 0.07 0.07" rgba="{r:.2f} {g:.2f} {b:.2f} 1" contype="1" conaffinity="1"/>'
            )

    xml = f'''
<mujoco model="random_scene">
  <include file="panda.xml"/>
  <option timestep="0.005" gravity="0 0 -9.81"/>
  <visual>
    <headlight diffuse="0.4 0.4 0.4" ambient="0.2 0.2 0.2" specular="0.3 0.3 0.3"/>
  </visual>
  <worldbody>
    <light name="scene_key" pos="0 -0.5 2.8" dir="0 0.2 -1" directional="true"/>
    <geom name="floor" type="plane" size="10 10 0.05"
          rgba="{floor_r:.2f} {floor_g:.2f} {floor_b:.2f} 1"/>
    <geom name="table" type="box" pos="0 -0.15 0.35" size="0.55 0.8 0.05"
          rgba="0.65 0.50 0.35 1" contype="1" conaffinity="1"/>
    {target_geom}
    {obstacles}
    <camera name="overview" pos="0.8 -1.5 1.8" xyaxes="0.85 0.53 0 -0.35 0.56 0.75"/>
  </worldbody>
</mujoco>'''
    return xml, action_label

# 테스트
xml_t, lbl_t = generate_random_scene(seed=42)
img_t = render_panda_scene(xml_t, scene_name='random_test', robot_pos=(0.0, -0.7, 0.40))
plt.imshow(img_t)
plt.title(f'랜덤 씬 (seed=42) | GT: {lbl_t}')
plt.axis('off'); plt.show()
print('✓ generate_random_scene 등록 완료')


### 5-2. 전체 파이프라인 함수 (연습문제)

씬 생성 → 렌더링 → VLM 분류 → 결과 기록을 하나의 함수로 묶습니다.


In [ ]:
def run_pipeline(action_label=None, seed=None, prompt=None, verbose=True):
    """단일 씬에 대해 전체 파이프라인을 실행합니다.

    Returns:
        dict: {xml, img, gt, pred, correct, response}
    """
    if prompt is None:
        prompt = H.PROMPT_ACTION_JSON

    # TODO 1: generate_random_scene으로 XML과 GT 레이블 생성
    xml, gt = _______________

    # TODO 2: render_panda_scene으로 씬 이미지 렌더링
    img = _______________

    # TODO 3: smolvlm_query로 VLM 응답 얻기
    response = _______________

    # TODO 4: H.parse_action으로 행동 파싱 및 정오 판단
    pred = _______________
    correct = (pred == gt)

    if verbose:
        print(f"{'✓' if correct else '✗'} GT: {gt:8s} | 예측: {pred:8s}")
    return {'xml': xml, 'img': img, 'gt': gt, 'pred': pred,
            'correct': correct, 'response': response}

# 단일 실행 테스트
result = run_pipeline(seed=7)
plt.imshow(result['img'])
plt.title(f"GT: {result['gt']} | 예측: {result['pred']}")
plt.axis('off'); plt.show()


### 5-3. 10개 씬 배치 평가

10개 랜덤 씬에 대해 자동으로 평가합니다.


In [ ]:
# 씬 10개 자동 평가
from collections import defaultdict
N = 10
batch_results = []
for i in range(N):
    label = ['pick','push','reorder','ignore'][i % 4]
    res = run_pipeline(action_label=label, seed=i*13)
    batch_results.append(res)

accuracy = sum(r['correct'] for r in batch_results) / N
print(f'\n=== 배치 평가 결과 ===')
print(f'전체 정확도: {accuracy:.0%} ({sum(r["correct"] for r in batch_results)}/{N})')

class_stats = defaultdict(lambda: [0, 0])
for r in batch_results:
    class_stats[r['gt']][0] += r['correct']
    class_stats[r['gt']][1] += 1

print('\n클래스별 정확도:')
for lbl in ['pick','push','reorder','ignore']:
    c, t = class_stats[lbl]
    if t > 0: print(f'  {lbl:8s}: {c}/{t} ({c/t:.0%})')


In [ ]:
# 배치 결과 시각화
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, res in zip(axes.flatten(), batch_results):
    ax.imshow(res['img'])
    color = 'green' if res['correct'] else 'red'
    ax.set_title(f"GT:{res['gt']}\n예측:{res['pred']}", color=color, fontsize=9)
    for spine in ax.spines.values(): spine.set_edgecolor(color); spine.set_linewidth(2)
    ax.axis('off')
plt.suptitle(f'배치 평가 (n={N}, 정확도 {accuracy:.0%})', fontsize=14)
plt.tight_layout(); plt.show()


---
## SECTION 6  |  ✍️ 미니과제 — 실시간 MuJoCo + VLM 파이프라인

> 이 과제는 수업 후 각자 완성하여 **공유 폴더에 제출**합니다.

### 과제 설명

`realtime_pipeline(n_scenes, seed)` 함수를 완성하세요.

**구현해야 하는 기능:**

1. **씬 생성** : `generate_random_scene()`으로 n개 씬 생성 (도메인 랜덤화 적용)  
2. **VLM 분류** : 각 씬 이미지를 SmolVLM으로 분류  
3. **GT 비교** : GT 레이블과 비교하여 정오 기록  
4. **결과 저장** : JSON 파일로 저장 후 공유 폴더에 업로드 (`H.team_save`)  
5. **시각화** : 클래스별 정확도 막대 그래프 출력  

### 평가 기준

| 항목 | 배점 |
|------|------|
| 파이프라인 코드가 오류 없이 실행됨 | 40점 |
| n=20 씬에 대해 전체·클래스별 정확도 출력 | 30점 |
| 클래스별 정확도 막대 그래프 포함 | 20점 |
| 결과 JSON이 공유 폴더에 저장됨 | 10점 |

### 힌트

```python
def realtime_pipeline(n_scenes=20, seed=0):
    results = []
    for i in range(n_scenes):
        # 1. generate_random_scene(seed=seed+i)
        # 2. H.render_scene(...)
        # 3. smolvlm_query(img, H.PROMPT_ACTION_JSON)
        # 4. H.parse_action(response)
        # 5. results.append({...})
    # 6. 정확도 계산 및 출력
    # 7. json.dump 후 H.team_save(path, folder='results')
    # 8. 막대 그래프 시각화
    return results
```

### 연구와의 연결

이 파이프라인은 **9~10회차 도메인 랜덤화 & Trajectory 수집**의 직접적인 기반입니다.  
VLM이 행동을 정확히 분류할수록 → GT 레이블 자동 생성 신뢰도가 높아집니다.  
최종적으로 이 레이블이 `smoVLA` 파인튜닝 데이터셋의 `action_seq` 가 됩니다.


In [ ]:
# ✍️ 과제 템플릿 (수업 후 완성)
import json

def realtime_pipeline(n_scenes=20, seed=0):
    """실시간 MuJoCo + VLM 연동 파이프라인.

    Args:
        n_scenes (int): 평가할 씬 수
        seed     (int): 재현 가능성을 위한 랜덤 시드

    Returns:
        list[dict]: 각 씬의 결과 목록
    """
    results = []

    # TODO 1: n_scenes 개 씬 생성 + VLM 분류 루프 구현
    # for i in range(n_scenes):
    #     xml, gt = generate_random_scene(seed=seed+i)
    #     img = render_panda_scene(xml, scene_name=f'scene_{i}')
    #     response = smolvlm_query(img, H.PROMPT_ACTION_JSON)
    #     pred = H.parse_action(response)
    #     results.append({'gt': gt, 'pred': pred, 'correct': pred==gt})

    # TODO 2: 전체 정확도 및 클래스별 정확도 출력

    # TODO 3: 결과를 JSON 파일로 저장
    # log_path = '/content/pipeline_results.json'
    # with open(log_path, 'w') as f:
    #     json.dump(results, f, ensure_ascii=False, indent=2)
    # H.team_save(log_path, folder='results')

    # TODO 4: 클래스별 정확도 막대 그래프 시각화

    return results


# 실행 (구현 완료 후 주석 해제)
# results = realtime_pipeline(n_scenes=20, seed=42)


---
## SECTION 9  |  오늘 배운 것 정리

| 개념 | 핵심 내용 |
|------|----------|
| VLM 아키텍처 | Vision Encoder → Projection → LLM — 이미지와 텍스트를 같은 공간에서 처리 |
| SmolVLM | 2B 파라미터, T4 GPU에서 실행 가능, smoVLA 비전 백본과 동일 |
| 프롬프트 전략 | Simple / JSON / Detailed — 정확도·속도 트레이드오프 존재 |
| 파이프라인 | 씬 생성 → 렌더링 → VLM 분류 → GT 비교 → 결과 저장 |

**다음 회차 예고**: 도메인 랜덤화 스크립트 — 더 다양한 씬을 자동 생성하고 5종 가림 시나리오를 구현합니다.
